In [0]:
%sql
use catalog deltalake_catalog;

In [0]:
%sql
create volume if not exists deltalake_catalog.default.deltalake_volume;

In [0]:
dbutils.fs.mkdirs("/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1")

True

In [0]:
from decimal import Decimal
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType, DecimalType

# Sample data with Decimal for unit_price
data = [
    (1, "SKU-1001", "Wireless Mouse", "Electronics", 2, Decimal("799.00")),
    (2, "SKU-2001", "Yoga Mat", "Fitness", 1, Decimal("1199.00")),
    (3, "SKU-3001", "Notebook A5", "Stationery", 5, Decimal("49.50")),
    (4, "SKU-4001", "Coffee Mug", "Kitchen", 3, Decimal("299.00")),
    (5, "SKU-5001", "LED Bulb", "Electronics", 4, Decimal("149.99"))
]

schema = StructType([
    StructField("order_id", LongType(), False),
    StructField("sku", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("product_category", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("unit_price", DecimalType(10,2), True) 
])

df = spark.createDataFrame(data, schema=schema)

display(df)
df.printSchema()

order_id,sku,product_name,product_category,qty,unit_price
1,SKU-1001,Wireless Mouse,Electronics,2,799.00
2,SKU-2001,Yoga Mat,Fitness,1,1199.00
3,SKU-3001,Notebook A5,Stationery,5,49.50
4,SKU-4001,Coffee Mug,Kitchen,3,299.00
5,SKU-5001,LED Bulb,Electronics,4,149.99


root
 |-- order_id: long (nullable = false)
 |-- sku: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)



In [0]:
volume_path = "/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1"
df.write.format("delta").mode("overwrite").save(volume_path)

In [0]:
%sql
DESCRIBE HISTORY DELTA.`/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2025-09-10T06:42:02.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(2134047553962488),0910-063422-brywvkx7-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> 5, numOutputBytes -> 1954)",null,Databricks-Runtime/17.1.x-photon-scala2.13


In [0]:
%sql
select * from DELTA.`/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1`

order_id,sku,product_name,product_category,qty,unit_price
1,SKU-1001,Wireless Mouse,Electronics,2,799.00
2,SKU-2001,Yoga Mat,Fitness,1,1199.00
3,SKU-3001,Notebook A5,Stationery,5,49.50
4,SKU-4001,Coffee Mug,Kitchen,3,299.00
5,SKU-5001,LED Bulb,Electronics,4,149.99


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_batch AS
SELECT * FROM VALUES
  (2, 'SKU-2001', 'Yoga Mat - updated', 'Fitness', 2, 1099.00),
  (5, 'SKU-5001', 'LED Bulb - change', 'Electronics', 6, 129.99),
  (6, 'SKU-6001', 'Travel Bottle', 'Accessories', 1, 249.00)
AS t(order_id, sku, product_name, product_category, qty, unit_price);


In [0]:
%sql
MERGE INTO DELTA.`/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1` as target
USING incoming_batch AS src
ON target.order_id = src.order_id
WHEN MATCHED THEN
UPDATE SET 
target.sku = src.sku,
target.product_name = src.product_name,
target.product_category = src.product_category,
target.qty = src.qty,
target.unit_price = src.unit_price
WHEN NOT MATCHED THEN
INSERT (order_id, sku, product_name, product_category, qty, unit_price)
VALUES (src.order_id, src.sku, src.product_name, src.product_category, src.qty, src.unit_price)


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,2,0,1


In [0]:
%sql
select * from DELTA.`/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1`

order_id,sku,product_name,product_category,qty,unit_price
1,SKU-1001,Wireless Mouse,Electronics,2,799.00
3,SKU-3001,Notebook A5,Stationery,5,49.50
4,SKU-4001,Coffee Mug,Kitchen,3,299.00
2,SKU-2001,Yoga Mat - updated,Fitness,2,1099.00
5,SKU-5001,LED Bulb - change,Electronics,6,129.99
6,SKU-6001,Travel Bottle,Accessories,1,249.00


In [0]:
%sql
DESCRIBE HISTORY DELTA.`/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2025-09-10T06:48:49.000Z,147711536460494,sumit.trendytech03@outlook.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2134047553962488),0910-063422-brywvkx7-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 4, numRemovedBytes -> 7155, p25FileSize -> 2049, numDeletionVectorsRemoved -> 1, minFileSize -> 2049, numAddedFiles -> 1, maxFileSize -> 2049, p75FileSize -> 2049, p50FileSize -> 2049, numAddedBytes -> 2049)",null,Databricks-Runtime/17.1.x-photon-scala2.13
1,2025-09-10T06:48:47.000Z,147711536460494,sumit.trendytech03@outlook.com,MERGE,"Map(predicate -> [""(order_id#11513L = cast(order_id#11501 as bigint))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2134047553962488),0910-063422-brywvkx7-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 3, numTargetBytesAdded -> 5201, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3301, materializeSourceTimeMs -> 221, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1669, numTargetRowsUpdated -> 2, numOutputRows -> 3, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1365)",null,Databricks-Runtime/17.1.x-photon-scala2.13
0,2025-09-10T06:42:02.000Z,147711536460494,sumit.trendytech03@outlook.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(2134047553962488),0910-063422-brywvkx7-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> 5, numOutputBytes -> 1954)",null,Databricks-Runtime/17.1.x-photon-scala2.13


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_flag AS
SELECT * FROM VALUES
  (2, 'SKU-2001', 'Yoga Mat - updated', 'Fitness', 2, 1021.00, FALSE),
  (3, 'SKU-3001', 'Notebook A5',      'Stationery', 5, 49.50,   TRUE)
AS t(order_id, sku, product_name, product_category, qty, unit_price, is_deleted);

In [0]:
%sql
MERGE INTO DELTA.`/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1` as target
USING incoming_flag AS src
ON target.order_id = src.order_id
WHEN MATCHED AND src.is_deleted = true THEN
DELETE
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,1,0


In [0]:
%sql
select * from DELTA.`/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1`

order_id,sku,product_name,product_category,qty,unit_price
1,SKU-1001,Wireless Mouse,Electronics,2,799.00
4,SKU-4001,Coffee Mug,Kitchen,3,299.00
5,SKU-5001,LED Bulb - change,Electronics,6,129.99
6,SKU-6001,Travel Bottle,Accessories,1,249.00
2,SKU-2001,Yoga Mat - updated,Fitness,2,1021.00


In [0]:
%sql
-- snapshot (only these rows should remain after the MERGE)
CREATE OR REPLACE TEMP VIEW incoming_snapshot AS
SELECT * FROM VALUES
  (1, 'SKU-1001', 'Wireless Mouse', 'Electronics', 2, 799.00),
  (2, 'SKU-2001', 'Yoga Mat - snapshot', 'Fitness', 3, 1049.00)
AS t(order_id, sku, product_name, product_category, qty, unit_price);

In [0]:
%sql
MERGE INTO DELTA.`/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1` as target
USING incoming_snapshot AS src
ON target.order_id = src.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE;


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,2,3,0


In [0]:
%sql
select * from DELTA.`/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1`

order_id,sku,product_name,product_category,qty,unit_price
1,SKU-1001,Wireless Mouse,Electronics,2,799.00
2,SKU-2001,Yoga Mat - snapshot,Fitness,3,1049.00


In [0]:
%sql
DESCRIBE HISTORY DELTA.`/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2025-09-10T06:57:34.000Z,147711536460494,sumit.trendytech03@outlook.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2134047553962488),0910-063422-brywvkx7-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 5441, p25FileSize -> 1834, numDeletionVectorsRemoved -> 1, minFileSize -> 1834, numAddedFiles -> 1, maxFileSize -> 1834, p75FileSize -> 1834, p50FileSize -> 1834, numAddedBytes -> 1834)",null,Databricks-Runtime/17.1.x-photon-scala2.13
5,2025-09-10T06:57:33.000Z,147711536460494,sumit.trendytech03@outlook.com,MERGE,"Map(predicate -> [""(order_id#12937L = cast(order_id#12925 as bigint))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [{""actionType"":""delete""}], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2134047553962488),0910-063422-brywvkx7-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 3, numTargetFilesAdded -> 2, numTargetBytesAdded -> 3447, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 2002, materializeSourceTimeMs -> 47, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 839, numTargetRowsUpdated -> 2, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 3, rewriteTimeMs -> 1086)",null,Databricks-Runtime/17.1.x-photon-scala2.13
4,2025-09-10T06:53:03.000Z,147711536460494,sumit.trendytech03@outlook.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2134047553962488),0910-063422-brywvkx7-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3810, p25FileSize -> 1994, numDeletionVectorsRemoved -> 1, minFileSize -> 1994, numAddedFiles -> 1, maxFileSize -> 1994, p75FileSize -> 1994, p50FileSize -> 1994, numAddedBytes -> 1994)",null,Databricks-Runtime/17.1.x-photon-scala2.13
3,2025-09-10T06:53:02.000Z,147711536460494,sumit.trendytech03@outlook.com,MERGE,"Map(predicate -> [""(order_id#12305L = cast(order_id#12292 as bigint))""], clusterBy -> [], matchedPredicates -> [{""predicate"":""is_deleted#12298: boolean"",""actionType"":""delete""},{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2134047553962488),0910-063422-brywvkx7-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 1, numTargetFilesAdded -> 1, numTargetBytesAdded -> 1761, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 1879, materializeSourceTimeMs -> 54, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 1, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 862, numTargetRowsUpdated -> 1, numOutputRows -> 1, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 930)",null,Databricks-Runtime/17.1.x-photon-scala2.13
2,2025-09-10T06:48:49.000Z,147711536460494,sumit.trendytech03@outlook.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2134047553962488),0910-063422-brywvkx7-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 4, numRemovedBytes -> 7155, p25FileSize -> 2049, numDeletionVectorsRemoved -> 1, minFileSize -> 2049, numAddedFiles -> 1, maxFileSize -> 2049, p75F

In [0]:

from delta.tables import DeltaTable

snapshot_rows = [
    (1, 'SKU-1001', 'Wireless Mouse', 'Electronics', 2, 799.00),
    (2, 'SKU-2001', 'Yoga Mat - snapshot', 'Fitness',    3, 1049.00)
]

schema = "order_id LONG, sku STRING, product_name STRING, product_category STRING, qty INT, unit_price DOUBLE"

incoming_snapshot_df = spark.createDataFrame(snapshot_rows, schema=schema)

path = "/Volumes/deltalake_catalog/default/delta_volume1/ordersdata1"
delta_table = DeltaTable.forPath(spark, path)

(delta_table.alias("t")
  .merge(incoming_snapshot_df.alias("s"), "t.order_id = s.order_id")
  .whenMatchedUpdateAll()
  .whenNotMatchedInsertAll()
  .whenNotMatchedBySourceDelete()
  .execute()
)

display(spark.sql(f"SELECT * FROM DELTA.`{path}` ORDER BY order_id"))
display(spark.sql(f"DESCRIBE HISTORY DELTA.`{path}` LIMIT 10"))

order_id,sku,product_name,product_category,qty,unit_price
1,SKU-1001,Wireless Mouse,Electronics,2,799.00
2,SKU-2001,Yoga Mat - snapshot,Fitness,3,1049.00


version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2025-09-10T07:01:01.000Z,147711536460494,sumit.trendytech03@outlook.com,MERGE,"Map(predicate -> [""(order_id#13845L = order_id#13869L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [{""actionType"":""delete""}], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2134047553962488),0910-063422-brywvkx7-v2n,6,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 3530, numTargetBytesRemoved -> 1834, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 1788, materializeSourceTimeMs -> 48, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 832, numTargetRowsUpdated -> 2, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 881)",null,Databricks-Runtime/17.1.x-photon-scala2.13
6,2025-09-10T06:57:34.000Z,147711536460494,sumit.trendytech03@outlook.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2134047553962488),0910-063422-brywvkx7-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 5441, p25FileSize -> 1834, numDeletionVectorsRemoved -> 1, minFileSize -> 1834, numAddedFiles -> 1, maxFileSize -> 1834, p75FileSize -> 1834, p50FileSize -> 1834, numAddedBytes -> 1834)",null,Databricks-Runtime/17.1.x-photon-scala2.13
5,2025-09-10T06:57:33.000Z,147711536460494,sumit.trendytech03@outlook.com,MERGE,"Map(predicate -> [""(order_id#12937L = cast(order_id#12925 as bigint))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [{""actionType"":""delete""}], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2134047553962488),0910-063422-brywvkx7-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 3, numTargetFilesAdded -> 2, numTargetBytesAdded -> 3447, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 2002, materializeSourceTimeMs -> 47, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 839, numTargetRowsUpdated -> 2, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 3, rewriteTimeMs -> 1086)",null,Databricks-Runtime/17.1.x-photon-scala2.13
4,2025-09-10T06:53:03.000Z,147711536460494,sumit.trendytech03@outlook.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2134047553962488),0910-063422-brywvkx7-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3810, p25FileSize -> 1994, numDeletionVectorsRemoved -> 1, minFileSize -> 1994, numAddedFiles -> 1, maxFileSize -> 1994, p75FileSize -> 1994, p50FileSize -> 1994, numAddedBytes -> 1994)",null,Databricks-Runtime/17.1.x-photon-scala2.13
3,2025-09-10T06:53:02.000Z,147711536460494,sumit.trendytech03@outlook.com,MERGE,"Map(predicate -> [""(order_id#12305L = cast(order_id#12292 as bigint))""], clusterBy -> [], matchedPredicates -> [{""predicate"":""is_deleted#12298: boolean"",""actionType"":""delete""},{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2134047553962488),0910-063422-brywvkx7-v2n,2,WriteSer